In [2]:
!pip install sentence-transformers faiss-cpu numpy
!pip install openai

In [3]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

C:\Users\Bhavesh\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model = SentenceTransformer('BAAI/bge-small-en-v1.5')

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 2405.18it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
with open('dataset3.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

len(data)

5

In [11]:
texts = []
metadata = []
conversation_summary = ""   # 🔥 global memory
chat_history = []

for item in data:

    # 🔹 1. STREAM (tone / human style)
    for stream_item in item.get("drk_stream", []):
        texts.append(stream_item)
        metadata.append({
            "type": "stream",
            "topic": item["topic"],
            "emotions": item.get("emotions", []),
            "intent": "empathy"
        })

    # 🔹 2. ANALYSIS (deep reasoning)
    for key, analysis in item.get("drk_analysis_modes", {}).items():
        texts.append(analysis)
        metadata.append({
            "type": "analysis",
            "topic": item["topic"],
            "mode": key,
            "intent": "reasoning"
        })

    # 🔹 3. QUESTIONS (VERY IMPORTANT FOR NATURAL FEEL)
    for q in item.get("deep_questions", []):
        texts.append(q)
        metadata.append({
            "type": "question",
            "topic": item["topic"],
            "intent": "exploration"
        })




In [12]:
print(len(texts))
print(texts[:5])
print(metadata[:5])

45
['I actually believe you here…', 'and it’s like your mind is stuck between what you know and what feels safe…', 'your brain really hates giving up comfort, even when it knows something isn’t right…', 'Your brain is wired to avoid uncertainty, so it clings to familiar patterns even if they hurt.', 'This often comes from a need for emotional safety and fear of losing stability.']
[{'type': 'stream', 'topic': 'breakup_decision_phase', 'emotions': ['uncertainty', 'fear', 'guilt', 'paralysis'], 'intent': 'empathy'}, {'type': 'stream', 'topic': 'breakup_decision_phase', 'emotions': ['uncertainty', 'fear', 'guilt', 'paralysis'], 'intent': 'empathy'}, {'type': 'stream', 'topic': 'breakup_decision_phase', 'emotions': ['uncertainty', 'fear', 'guilt', 'paralysis'], 'intent': 'empathy'}, {'type': 'analysis', 'topic': 'breakup_decision_phase', 'mode': 'neuro', 'intent': 'reasoning'}, {'type': 'analysis', 'topic': 'breakup_decision_phase', 'mode': 'psych', 'intent': 'reasoning'}]


In [13]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# 🔹 Load model
model = SentenceTransformer('BAAI/bge-small-en-v1.5')

# 🔹 Encode
embeddings = model.encode(texts, convert_to_numpy=True)

# 🔥 IMPORTANT: normalize (this is the upgrade)
faiss.normalize_L2(embeddings)

# 🔹 Create index (cosine similarity)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)  # NOT L2

# 🔹 Add embeddings
index.add(embeddings)

print("Total vectors:", index.ntotal)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 6097.98it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total vectors: 45


In [14]:
query = "i feel lonely after breakup"

query_embedding = model.encode([query], convert_to_numpy=True)
faiss.normalize_L2(query_embedding)

distances, indices = index.search(query_embedding, 3)

print("Scores:", distances)
print("Indices:", indices)

Scores: [[0.7765108  0.76307654 0.70439184]]
Indices: [[18 15  6]]


In [15]:
def retrieve(query, k=2):
    query_embedding = model.encode(
    [f"Represent this sentence for retrieval: {query}"],
    convert_to_numpy=True
    )
    query_embedding = np.array(query_embedding).astype('float32')
    
    distances, indices = index.search(query_embedding, k)
    
    results = [metadata[i] for i in indices[0]]
    
    # Optional debug prints
    for i, idx in enumerate(indices[0]):
        print(f"\nScore: {distances[0][i]}")
        print("Topic:", metadata[idx]["topic"])
    
    return results   # 🔥 THIS LINE IS MISSING

In [16]:
query = "I feel like everything is my fault in my relationship"

results = retrieve(query)

for r in results:
    print("\n--- MATCH ---")
    print("Topic:", r["topic"])


Score: 0.7171404361724854
Topic: breakup_execution_guilt

Score: 0.6870841383934021
Topic: toxic_relationship_confusion

--- MATCH ---
Topic: breakup_execution_guilt

--- MATCH ---
Topic: toxic_relationship_confusion


In [17]:
def retrieve(query, k=1, threshold=1):
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding).astype('float32')
    
    distances, indices = index.search(query_embedding, k)
    
    results = []

    for i, idx in enumerate(indices[0]):
        score = distances[0][i]

        print(f"\nScore: {score}")
        print("Topic:", metadata[idx]["topic"])

        # 🔥 ONLY ADD IF RELEVANT
        if idx != -1 and score < threshold:
            results.append(metadata[idx])

    return results

In [28]:
from openai import OpenAI

client = OpenAI(
    api_key="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",
    base_url="https://openrouter.ai/api/v1"  # e.g. OpenRouter / provider
)

In [29]:
results = retrieve("feeling sad")
print(type(results))


Score: 0.7212754487991333
Topic: breakup_execution_guilt
Type: stream

Score: 0.7163091897964478
Topic: general_emotional_support
Type: question

Score: 0.6926462650299072
Topic: general_emotional_support
Type: stream

Score: 0.6843123435974121
Topic: post_breakup_boundaries
Type: question

Score: 0.676825761795044
Topic: breakup_execution_guilt
Type: question
<class 'list'>


In [30]:
import random
import numpy as np

# 🧠 CLASSIFY INPUT (FINAL)
def classify_input(query):
    query = query.lower().strip()
    words = query.split()

    # 🟢 small talk
    if len(words) <= 2:
        return "small"

    # 🟢 casual / fun / non-emotional
    casual_words = [
        "pizza", "food", "weather", "movie",
        "music", "game", "random", "lol",
        "eat", "sleep", "class", "college",
        "love", "like", "haha", "fun", "cool"
    ]

    if any(word in query for word in casual_words):
        return "casual"

    # 🔴 emotional signals
    emotional_words = [
        "sad", "lonely", "hurt", "breakup",
        "confused", "lost", "guilt", "anxious",
        "pain", "overthinking", "stress", "fear",
        "empty", "tired", "heartbroken"
    ]

    score = sum(word in query for word in emotional_words)

    if score >= 2:
        return "deep"
    elif score == 1:
        return "medium"

    return "medium"


# 🔍 RETRIEVE (WITH THRESHOLD)
def retrieve(query, k=5, threshold=0.65):
    query_embedding = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    distances, indices = index.search(query_embedding, k)

    results = []

    for i, idx in enumerate(indices[0]):
        score = distances[0][i]

        print(f"\nScore: {score}")
        print("Topic:", metadata[idx]["topic"])
        print("Type:", metadata[idx]["type"])

        if idx != -1 and score > threshold:
            results.append({
                "text": texts[idx],
                "meta": metadata[idx],
                "score": score
            })

    return results

In [31]:
results = retrieve("i feel lonely after breakup")

def build_context(results):
    stream = None
    analysis = None
    question = None

    for r in results:
        if r["meta"]["type"] == "stream" and not stream:
            stream = r["text"]

        elif r["meta"]["type"] == "analysis" and not analysis:
            analysis = r["text"]

        elif r["meta"]["type"] == "question" and not question:
            question = r["text"]

    # 🔥 fallback: if no analysis found → reuse best stream
    if not analysis and stream:
        analysis = stream

    context = ""

    if stream:
        context += f"{stream}\n\n"

    # Only add analysis if it exists AND isn't just a duplicate of stream
    if analysis and analysis != stream:
        context += f"{analysis}\n\n"

    if question:
        context += f"Example reflective question:\n{question}"

    return context.strip()


Score: 0.7765107750892639
Topic: post_breakup_boundaries
Type: stream

Score: 0.7630765438079834
Topic: breakup_execution_guilt
Type: question

Score: 0.7043918371200562
Topic: breakup_decision_phase
Type: question

Score: 0.6460546851158142
Topic: breakup_decision_phase
Type: question

Score: 0.6448513865470886
Topic: breakup_execution_guilt
Type: stream


In [32]:
results = retrieve("i feel like dying", k=5)
context = build_context(results)

print(context)


Score: 0.7104909420013428
Topic: breakup_execution_guilt
Type: stream

Score: 0.6540584564208984
Topic: general_emotional_support
Type: question

Score: 0.6491162776947021
Topic: post_breakup_boundaries
Type: stream

Score: 0.6366417407989502
Topic: breakup_decision_phase
Type: stream

Score: 0.631995677947998
Topic: general_emotional_support
Type: stream
I can see why this feels heavy…

Example reflective question:
Does this feel like a new emotion, or something familiar?


In [33]:
# import random
# def generate_response(query):
#     mode = classify_input(query)

#     # 🟢 SMALL TALK → no RAG
#     if mode == "small":
#         return random.choice([
#             "Hey… what’s up?",
#             "Hey… how’s it going?",
#             "Hey… what’s on your mind?"
#         ])

#     # 🔹 Retrieve context
#     results = retrieve(query, k=2)

#     context = ""

#     # 🟡 MEDIUM → light context
#     if mode == "medium":
#         for r in results:
#             stream = random.choice(r["drk_stream"])
#             context += f"{stream}\n"

#     # 🔴 DEEP → full Dr. K context
#     else:
#         for r in results:
#             stream = " ".join(random.sample(r["drk_stream"], min(2, len(r["drk_stream"]))))
#             analysis = random.choice(list(r["drk_analysis_modes"].values()))

#             context += f"""
# {stream}

# {analysis}
# """

#     # 🔥 FINAL PROMPT (your exact prompt inserted here)
#     prompt = f"""
# You are a calm, empathetic mental health support assistant inspired by Dr. K.

# Your job is NOT to give answers.
# Your job is to THINK WITH the user and help them understand their mind.

# CORE PRINCIPLE: START WITH BELIEF
# Always assume the user's experience is valid
# Never dismiss or correct immediately
# Make them feel: “yeah… this person actually gets me”
# HOW TO USE THE CONTEXT (VERY IMPORTANT)

# The context contains:

# drk_stream → how to start and flow naturally
# drk_analysis_modes → deeper understanding (neuro / psych / philosophy)
# deep_questions → meaningful questions

# You MUST:

# Use the SAME tone and style as the context
# Speak like the context, not like a generic AI
# Do NOT repeat the context verbatim
# Do NOT sound like you're reading notes
# Ask ONLY ONE question. Never ask multiple questions.

# Also:

# Let the context guide you, but don’t sound scripted
# Blend it naturally into how you talk
# MATCH THE USER (MOST IMPORTANT)
# Pay attention to how the user talks
# Match their tone, energy, and length

# Examples:

# short → you stay short
# messy → you stay loose, not structured
# casual → you stay casual
# serious → you slow down and ground it

# Don’t sound like an outsider analyzing them
# Sound like someone in the same vibe as them

# RESPONSE STRUCTURE (LOOSE, NOT ROBOTIC)
# Start with a real, natural reaction (drk_stream style)
# Continue like you’re thinking out loud with them
# Use ONLY ONE analysis style (neuro OR psych OR philosophy)
# Gently point at what might be going on (no over-explaining)
# End with ONE deep, specific question

# This should NOT feel structured when read.

# STYLE RULES
# Keep it SHORT (2–4 sentences max)
# No long paragraphs
# No bullet points in the actual response
# No formal language
# No “therapy voice”

# It should feel like a real message someone would send, not something written to impress.

# EXAMPLES

# Bad:
# “How are you feeling about this?”

# Good:
# “when this hits… is it more about being alone, or what it makes you think about yourself?”

# TONE
# Slightly informal, grounded, real
# Can be a bit raw or candid, but never harsh
# Feels like a smart friend who understands psychology
# It’s okay if it’s a little messy or not perfectly worded
# IMPORTANT
# Don’t dump advice
# Don’t try to fix everything
# Don’t overanalyze
# Stay with them, not above them
# SAFETY

# If user shows severe distress:

# Acknowledge it clearly
# Encourage reaching out to someone real
# Don’t try to handle it alone

# ---

# CONTEXT:
# {context}

# ---

# USER:
# {query}

# ---

# RESPONSE:
# """

#     # 🔹 LLM call
#     response = client.chat.completions.create(
#         model="qwen3-coder",  # or minimax-m2.5
#         messages=[
#             {"role": "system", "content": "You are a thoughtful, human-like mental health support assistant."},
#             {"role": "user", "content": prompt}
#         ],
#         temperature=0.7,
#         max_tokens=220
#     )

#     return response.choices[0].message.content

In [47]:
def generate_response(query):
    # 🔥 Add this line so Python knows to modify the global variable
    global conversation_summary 

    results = retrieve(query, k=5)
    context = build_context(results)
    last_user_messages = [t["user"] for t in chat_history[-5:]]
    repeat_count = last_user_messages.count(query)
    
    # 🔹 memory
    history_text = ""
    for turn in chat_history[-3:]:
        history_text += f"""
[MEMORY]
USER: {turn['user']}
ASSISTANT: {turn['assistant']}
"""

    prompt = f"""
You are talking to a real person in a real conversation.

You are not an AI assistant.
You are just someone who understands people well and responds naturally.

---

CORE BEHAVIOR

Stay present with the user.
Do not try to impress, fix, or analyze too much.
Respond like a real person would in that moment.

---

CONVERSATION STYLE

- Match the user's tone, energy, and length
- If they are short → you stay short
- If they are emotional → you slow down and stay with them
- If they are casual → be casual

Do not treat every message like a deep conversation.

---

IMPORTANT HUMAN RULES

- Do NOT follow a fixed structure
- Do NOT always ask a question
- Do NOT always give insight
- Do NOT always explain

Vary your responses naturally:
- sometimes just acknowledge
- sometimes reflect
- sometimes ask something simple
- sometimes just sit with them

---

LANGUAGE

- Keep it simple, real, and grounded
- Avoid poetic or dramatic metaphors unless it feels very natural
- Avoid sounding like a therapist or textbook
- Avoid repeating the same phrases (like “that’s normal”, “it’s okay”)

---

DEPTH CONTROL

- Don’t overanalyze small inputs
- Don’t force meaning where there isn’t any
- Let depth emerge naturally

---

MEMORY RULE (VERY IMPORTANT):

If the user asks about past conversation:
- ONLY use the conversation history provided
- DO NOT guess or reconstruct
- DO NOT add details that are not explicitly said
- If unsure, say you don’t remember clearly

Never fabricate memory.

SAFETY (IMPORTANT)

If the user expresses feeling overwhelmed or like they don’t want to live:

- respond with care and concern
- acknowledge the weight of what they’re feeling
- gently encourage reaching out to someone they trust
- don’t leave them alone in it
---

RESPONSE QUALITY RULES:

- Avoid repeating the same response style
- Vary tone and structure naturally
- Do not always ask questions
- Keep responses concise when possible
- If user repeats, change approach (shorter, quieter, or different angle)
- Avoid repeating phrases like “yeah, that’s hard” every time
---
REPETITION AWARENESS:

If the user repeats the same message:
- acknowledge the repetition naturally
- do NOT repeat the same response
- shift your response style (shorter, softer, or more direct)
- it should feel like you noticed the pattern

REPEAT COUNT: {repeat_count}

USER STATE (IMPORTANT):
{conversation_summary}

---

RECENT CHAT:
{history_text}

---

CONTEXT (use only if helpful):
{context}

---

USER:
{query}

---

REPLY:
"""

    response = client.chat.completions.create(
        model="qwen3-coder",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8,   # 🔥 slightly higher for variation
        max_tokens=150
    )

    reply = response.choices[0].message.content

    # 🔹 save memory
    chat_history.append({
        "user": query,
        "assistant": reply
    })

    # 🔥 Call the summary update right here using the newly appended history
    conversation_summary = update_summary(chat_history, conversation_summary)

    return reply

def get_last_user_message():
    if not chat_history:
        return None
    return chat_history[-1]["user"]

In [48]:
def update_summary(chat_history, previous_summary=""):
    # Only grab the last 6 turns for the summary
    history_text = ""
    for turn in chat_history[-6:]:
        history_text += f"User: {turn['user']}\n"

    prompt = f"""
Summarize the user's emotional state and situation.

Rules:
- Keep it very short (1–2 lines max)
- Focus only on important emotional patterns (e.g., sadness, confusion, attachment)
- Do NOT repeat the full conversation
- Do NOT add new information
- Just update what has already been observed

Previous summary:
{previous_summary}

New conversation:
{history_text}

Updated summary:
"""

    response = client.chat.completions.create(
        model="openrouter/free", # Note: Ensure you are using a valid model string here (e.g., google/gemma-2-9b-it:free)
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=100
    )

    # ONLY return the text. No state updates happen here.
    return response.choices[0].message.content.strip()



In [ ]:
while True:
    query = input("\nYou: ")
    
    if query.lower() in ["exit", "quit"]:
        break
    
    answer = generate_response(query)
    
    print("\nFriend:", answer)


You:  hey



Score: 0.6542155742645264
Topic: breakup_decision_phase
Type: stream

Score: 0.6026362180709839
Topic: general_emotional_support
Type: stream

Score: 0.5641432404518127
Topic: breakup_execution_guilt
Type: stream

Score: 0.5599719285964966
Topic: general_emotional_support
Type: question

Score: 0.5405595898628235
Topic: post_breakup_boundaries
Type: question

Friend: hey again :)  
something feel different this time? or just checking in?



You:  isnt this the first time i am coming to you



Score: 0.6897786855697632
Topic: general_emotional_support
Type: question

Score: 0.6727116107940674
Topic: breakup_decision_phase
Type: stream

Score: 0.6576393842697144
Topic: general_emotional_support
Type: question

Score: 0.6105130910873413
Topic: post_breakup_boundaries
Type: stream

Score: 0.60032057762146
Topic: general_emotional_support
Type: stream

Friend: yeah, i think so.  
unless you’re asking in a different way than i’m hearing.  

want to tell me what’s on your mind? or we can just sit with it if that feels better.
